# 02 — Bronze des taux de change BCE (Auto Loader, CSV)

Lit les fichiers `exr.csv` déposés par `01_ingest_ecb` dans `/Volumes/supplychain/landing/erp/fx`
et les ajoute dans la table Delta `bronze.ecb_exchange_rates`.

- une ligne = une devise × un jour publié × un run ; valeurs gardées en texte (typage en Silver) ;
- doublons conservés (les runs se chevauchent de 10 jours) : le Silver garde un taux par devise et par jour ;
- incrémental : le checkpoint mémorise les fichiers déjà lus ; `availableNow` traite le nouveau puis s'arrête.

Exécuté par le Job `agrimeca_fx_daily`, tâche `bronze_fx`, après la réussite de `ingest_ecb`.

In [ ]:
# Cellule 1 - paramètres
dbutils.widgets.text("catalog", "supplychain")
dbutils.widgets.dropdown("reset", "non", ["non", "oui"])   # "oui" = reconstruire la table de zéro

CATALOG = dbutils.widgets.get("catalog")
RESET = dbutils.widgets.get("reset") == "oui"

SOURCE = f"/Volumes/{CATALOG}/landing/erp/fx"
TABLE = f"{CATALOG}.bronze.ecb_exchange_rates"
CHECKPOINT = f"/Volumes/{CATALOG}/landing/checkpoints/ecb_exchange_rates"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"USE CATALOG {CATALOG}")

In [ ]:
# Cellule 2 - remise à zéro (seulement si reset = "oui") : table ET checkpoint ensemble
if RESET:
    spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
    dbutils.fs.rm(CHECKPOINT, recurse=True)
    print("table et checkpoint supprimés")
else:
    print("reset = non : chargement incrémental")

In [ ]:
# Cellule 3 - chargement incrémental
import time
from pyspark.sql import functions as F


def table_stats(table: str) -> tuple[int, int]:
    # (lignes, fichiers) de la table ; (0, 0) si elle n'existe pas encore
    if not spark.catalog.tableExists(table):
        return 0, 0
    row = spark.table(table).agg(F.count("*"), F.countDistinct("_file")).first()
    return row[0], row[1]


rows_before, files_before = table_stats(TABLE)
start = time.time()

rates = (spark.readStream
         .format("cloudFiles")                                    # Auto Loader
         .option("cloudFiles.format", "csv")
         .option("header", True)                                  # 1re ligne = noms de colonnes
         .option("cloudFiles.inferColumnTypes", False)            # tout en texte : pas de taux mal typé
         .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")  # schéma mémorisé entre deux runs
         .option("pathGlobFilter", "exr.csv")
         .option("recursiveFileLookup", True)                     # parcourt ingest_date=…/run_id=…
         .load(SOURCE))

# Le CSV BCE a une trentaine de colonnes : on garde les trois utiles + la traçabilité
records = (rates
           .withColumn("_file", F.col("_metadata.file_path"))
           .select(F.col("CURRENCY").alias("currency"),
                   F.col("TIME_PERIOD").alias("rate_date"),
                   F.col("OBS_VALUE").alias("rate"),
                   F.regexp_extract("_file", r"run_id=([^/]+)", 1).alias("_run_id"),
                   "_file",
                   F.current_timestamp().alias("_ingest_ts")))

query = (records.writeStream
         .queryName("bronze_ecb_exchange_rates")
         .option("checkpointLocation", CHECKPOINT)
         .trigger(availableNow=True)                              # tout ce qui est nouveau, puis arrêt
         .toTable(TABLE))
query.awaitTermination()

rows_after, files_after = table_stats(TABLE)
print(f"ecb_exchange_rates : {files_after - files_before} fichiers lus, "
      f"{rows_after - rows_before} lignes ajoutées, {rows_after} au total, {time.time() - start:.0f} s")

## Contrôles

In [ ]:
%sql
-- Cellule 4 - par devise : nombre de jours, période couverte, taux extrêmes
SELECT currency,
       count(*)                       AS lignes,
       count(DISTINCT rate_date)      AS jours_distincts,
       min(rate_date)                 AS debut,
       max(rate_date)                 AS fin,
       min(cast(rate AS DOUBLE))      AS taux_min,
       max(cast(rate AS DOUBLE))      AS taux_max
FROM bronze.ecb_exchange_rates
GROUP BY currency
ORDER BY currency

In [ ]:
%sql
-- Cellule 5 - qualité technique : tout doit valoir 0
SELECT count_if(currency IS NULL)                  AS sans_devise,
       count_if(rate_date IS NULL)                 AS sans_date,
       count_if(try_cast(rate AS DOUBLE) IS NULL)  AS taux_non_numerique,
       count_if(_run_id = '')                      AS sans_run
FROM bronze.ecb_exchange_rates

In [ ]:
%sql
-- Cellule 6 - lignes par run (chaque exécution du Job ajoute un run)
SELECT _run_id, count(*) AS lignes, min(rate_date) AS debut, max(rate_date) AS fin
FROM bronze.ecb_exchange_rates
GROUP BY _run_id
ORDER BY _run_id DESC